# Process the refusals files
This script will process refusal files that have been summarized as note and date (or the whole file, when the original note file was empty)

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
from striprtf.striprtf import rtf_to_text
import re
from tqdm import tqdm
#tqdm.pandas() 
import swifter
from collections import defaultdict
import ast
from concurrent.futures import ProcessPoolExecutor
from functools import partial
import gzip
import datetime
import pytz

In [ ]:
# load other files for linkage
children = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/processed/patients_ruca_svi_2018.csv.zip')
politics = pd.read_csv('/share/pi/deho-pi/AFC/l2_afc_match_032125_using2018.csv')
#engaged_before_4 = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/engaged_before_4.csv.zip')
engage_date = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/earliest_engaged.csv.zip')

In [ ]:
# load data
# put in all of the AFC data
with gzip.open('/share/pi/deho-pi/AFC/BQ/GeneratedPatientBaseline_0724.csv.gz') as f:
    patient_baseline = pd.read_csv(f, low_memory = False)

In [ ]:
child_politics = children[children['patientuid'].isin(engage_date['patientuid'])]
child_politics = child_politics[~child_politics['party'].isna()]

# Read note-based refusals

In [ ]:
# list out the files
path = '/share/pi/deho/AFC/mortonc/intermediate/refusals_notes/'
file_head = 'refusal_notes'
filenames = os.listdir(path)


In [ ]:
results = pd.DataFrame({'patientuid':[], 'encounterdate':[]})

In [ ]:
# read in and join the note files
for filename in filenames:
    full_path = os.path.join(path, filename)
    df = pd.read_csv(full_path)

    if len(df) != 0:
        if (len(df.columns) == len(results.columns)) and (df.columns == results.columns).all():
            results = pd.concat([results, df])
            print(filename)
        else:
            print("Failed to process " + filename)
    

In [ ]:
min(results['encounterdate'])

In [ ]:
# save the file to intermediate
results.to_csv('/share/pi/deho/AFC/mortonc/intermediate/refusal_note_summary_2018.csv')

# Read in code-based refusals

In [ ]:
code_refus = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/vax_refusal.csv.zip')

In [ ]:
min(code_refus['observation_date'])

In [ ]:
code_refus['encounterdate'] = code_refus['observation_date']

In [ ]:
code_refus = code_refus[['patientuid', 'encounterdate']]

In [ ]:
code_refus['encounterdate'] = pd.to_datetime(code_refus['encounterdate']).dt.tz_localize('UTC')

In [ ]:
code_refus

# Concat code/ID- and note-based refusals, process

In [ ]:
len(results)

In [ ]:
results = pd.read_csv('/share/pi/deho/AFC/mortonc/intermediate/refusal_note_summary_2018.csv')

In [ ]:
results = results[['patientuid', 'encounterdate']]

In [ ]:
len(results)

In [ ]:
results = pd.concat([results, code_refus])

In [ ]:
len(results)

In [ ]:
# process results -- get the earliest refusal
results['encounterdate'] = pd.to_datetime(results['encounterdate'])
results = results.groupby('patientuid')['encounterdate'].min().reset_index()

In [ ]:
len(child_politics)

In [ ]:
child_politics = pd.merge(child_politics, results, how = 'left')

In [ ]:
len(child_politics)

In [ ]:
def to_ordinal_with_na(date):
    if pd.isna(date):
        return np.nan  # Return NaN for missing values
    else:
        return date.toordinal()
def myround(x, base=4):
    return base * round(x/base)

In [ ]:
def process_child_politics(child_politics):
    child_politics.loc[:,'dob'] = pd.to_datetime(child_politics['dob'], format='mixed')
    child_politics.loc[:,'dob_ordinal'] = [x.toordinal() for x in child_politics['dob']]

    child_politics.loc[:,'encounterdate'] = pd.to_datetime(child_politics['encounterdate'], format='mixed')

    child_politics.loc[:,'encounterdate_ordinal'] = [to_ordinal_with_na(x) for x in child_politics['encounterdate']]

    child_politics.loc[:,'time_to_refusal'] = (child_politics['encounterdate_ordinal'] - child_politics['dob_ordinal'])/365.0

    # fill in unknown child_politics values with 100, create indicator column with 0s
    child_politics['time_to_refusal'] = child_politics['time_to_refusal'].fillna(value=100)
    
    return(child_politics)

In [ ]:
child_politics = process_child_politics(child_politics)

Proportion of children in a household with Dem/Rep/Bipartisan affiliation who do not have a refusal on file. 

In [ ]:
sum(child_politics['encounterdate'].isna())/len(child_politics)

Proportion of children in a household with Dem/Rep/Bipartisan affiliation who have a refusal on file while age 4 or below

In [ ]:
sum(child_politics['time_to_refusal'] < 5)/len(child_politics)

## Add in date of first engagement

In [ ]:
child_politics = pd.merge(child_politics, engage_date, how = 'left')

In [ ]:
len(child_politics)

# Save refusals

In [ ]:
def save_zip_csv(filepath, dataset):
    # write to CSV
    csv_filename = filepath
    dataset.to_csv(csv_filename, index=False)

    # zip CSV
    zip_filename = csv_filename + '.zip'

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(csv_filename, os.path.basename(csv_filename))

    # remove large csv
    os.remove(csv_filename)
    
    print("Saved!")

In [ ]:
save_zip_csv('/share/pi/deho/AFC/mortonc/intermediate/refusals_codes_notes_2018.csv', child_politics)

In [ ]:
len(child_politics)

In [ ]:
sum(child_politics['party'] == 'Republican') + sum(child_politics['party'] == 'Democratic')

In [ ]:
child_politics.columns